# 07 - CNN-BiLSTM (character branch)

Learns character ordering, which the hand-crafted lexical features discard. Focal loss rather than SMOTE: interpolating between character embeddings would synthesise domains that cannot exist.

Stage data locally first - reading batches off the Drive mount will bottleneck the GPU.

In [ ]:
# --- standard header: every notebook starts with exactly this ---
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess, getpass
REPO = '/content/secure-dns-trust-ai'
URL  = 'github.com/sandesh20lamichhane/secure-dns-trust-ai.git'

if os.path.isdir(REPO):
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=False)
else:
    # Private repo: paste your GitHub Personal Access Token when prompted.
    # It is only held in this runtime and vanishes when the session ends.
    TOKEN = getpass.getpass('GitHub PAT: ')
    subprocess.run(['git', 'clone', '-q', f'https://{TOKEN}@{URL}', REPO], check=True)

sys.path.insert(0, REPO)
os.environ['DNSTRUST_CONFIG_DIR'] = f'{REPO}/configs'

from src.utils import config, manifest, seeds, io
P = config.paths(); config.ensure_tree(P); seeds.set_all(42)
print('repo', manifest.git_sha(REPO))


In [ ]:
import numpy as np, torch, pandas as pd
from torch.utils.data import TensorDataset, DataLoader
from src.models.cnn_bilstm import CharEncoder, CNNBiLSTM, FocalLoss
from src.evaluate import splits, metrics, predictions

cfg = config.load('cnn_bilstm')
local = io.stage_local(f"{P['data']['features']}/fused_v1.parquet", P['local']['data'])
df = pd.read_parquet(local)
split = splits.load_split(P['data']['splits'], cfg['split']['name'])
tr, va, te = splits.apply_split(df, split)

In [ ]:
enc = CharEncoder(cfg['input']['charset'], cfg['input']['max_length'])
def loader(frame, shuffle):
    X = torch.from_numpy(enc.encode_batch(frame['domain'].values))
    y = torch.tensor(frame['label'].values, dtype=torch.float32)
    return DataLoader(TensorDataset(X, y), batch_size=cfg['train']['batch_size'], shuffle=shuffle)

dl_tr, dl_va, dl_te = loader(tr, True), loader(va, False), loader(te, False)
model = CNNBiLSTM(enc.vocab_size, **{k: v for k, v in cfg['model'].items()}).cuda()
opt = torch.optim.AdamW(model.parameters(), lr=cfg['train']['lr'],
                        weight_decay=cfg['train']['weight_decay'])
crit = FocalLoss(gamma=cfg['train']['focal_gamma'])

In [ ]:
best, patience = -1, 0
for epoch in range(cfg['train']['epochs']):
    model.train()
    for xb, yb in dl_tr:
        opt.zero_grad(); loss = crit(model(xb.cuda()), yb.cuda()); loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        s = np.concatenate([torch.sigmoid(model(xb.cuda())).cpu().numpy() for xb, _ in dl_va])
    m = metrics.evaluate(va['label'].values, s)
    print(epoch, round(m['pr_auc'], 4))
    if m['pr_auc'] > best:
        best, patience = m['pr_auc'], 0
        torch.save(model.state_dict(), '/content/best_cnn_bilstm.pt')
    else:
        patience += 1
        if patience >= cfg['train']['early_stopping_patience']: break

In [ ]:
model.load_state_dict(torch.load('/content/best_cnn_bilstm.pt'))
with torch.no_grad():
    scores = np.concatenate([torch.sigmoid(model(xb.cuda())).cpu().numpy() for xb, _ in dl_te])
m = metrics.evaluate(te['label'].values, scores)

counter = manifest.next_counter(P['manifest'])
run_id = manifest.make_run_id('cnnbilstm', cfg['split']['name'], cfg['seed'], counter)
torch.save(model.state_dict(), f"{P['artifacts']['models']}/{run_id}.pt")
predictions.save(run_id, P['artifacts']['predictions'], te['domain'], te['label'].values, scores)
manifest.record(P['manifest'], run_id, 'cnn_bilstm', cfg, cfg['split']['name'],
                split['split_file'], m, cfg['seed'], repo_root=REPO)
print(run_id, m)